In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import os
import tensorflow as tf

2026-02-20 05:47:30.745206: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771595250.760711  507511 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771595250.765863  507511 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771595250.779988  507511 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771595250.780002  507511 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771595250.780004  507511 computation_placer.cc:177] computation placer alr

In [2]:
# minorized reference
with h5py.File('/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5', 'r') as file:
    #print("Keys: %s" % file.keys())
    ref_d = file['density'][:]
#print(ref_d)
print(ref_d.shape)




(116, 116, 72)


In [3]:
print(np.sum(ref_d))

50.0


In [4]:
#Transform density
def transform(density,density_ref,transform_type):
    dens = density
    dref  = density_ref
    ttype = transform_type
    if ttype=='value':
        dtrans = density
    elif ttype=='sqrt':
        dtrans = np.sqrt(np.abs(density))
    elif ttype=='log':
        dtrans  = np.log(np.abs(density))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        dtrans = (dens-dval)/dsqrt
    else:
        raise RuntimeError('bad transform type')
    return dtrans


def inverse_transform(density_trans,density_ref,transform_type):
    dref   = density_ref
    ttype  = transform_type
    dtrans = density_trans
    if ttype=='value':
        density = dtrans
    elif ttype=='sqrt':
        density = dtrans**2
    elif ttype=='log':
        density = np.log(np.abs(dtrans))
    elif ttype=='residual_noise':
        dval  = dref
        dsqrt = np.sqrt(np.abs(dref))
        density = dval + dsqrt*dtrans
    else:
        raise RuntimeError('bad transform type')
    return density


dft_path ='/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'
with h5py.File(dft_path, 'r') as file:
    dft_d = file['density'][:]

In [5]:
####################################################################################################################################################
def stochastic_density(d,N):
    # poisson model
    #  accurate and fast for all values of N
    # N  = number of MC samples
    assert isinstance(d,np.ndarray)
    assert isinstance(N,(int,float,np.int64,np.float64))
    assert N>0
    ds = np.random.poisson(N*d)/N
    ds*= d.sum()/ds.sum()
    return ds
#end def stochastic_density

####################################################################################################################################################

In [7]:
import numpy as np
import h5py
import scipy.ndimage

# ==========================================
# 1. SETUP & PARAMETERS
# ==========================================

samp = [165150720, 41287680, 20643840] 

# ---------------------------------------------------------------------
# NEW FUNCTION: Physics-Aware Perturbation
# ---------------------------------------------------------------------
def perturb_density(base_density, target_electrons=50.0):
    """
    Creates a synthetic "True Density" by adding perturbations that are:
    1. Proportional to the local density (so they follow the 'doughnuts').
    2. Renormalized to ensure the total integral equals 'target_electrons'.
    """
    # 1. Start with a copy
    distorted = base_density.copy()
    
    # 2. Generate Random Gaussian Field (The "Blobs")
    # We generate a low-res field and upsample it for smoothness
    # This creates large, coherent fluctuations rather than pixel noise
    field_shape = np.array(base_density.shape) // 4  # Lower res grid
    random_field = np.random.uniform(-1.0, 1.0, field_shape)
    
    # Zoom it back up to full size (Linear interpolation is smooth enough here)
    zoom_factors = np.array(base_density.shape) / np.array(random_field.shape)
    smooth_field = scipy.ndimage.zoom(random_field, zoom_factors, order=1)
    
    # Handle slight shape mismatches from rounding in zoom
    smooth_field = smooth_field[:base_density.shape[0], :base_density.shape[1], :base_density.shape[2]]
    
    # 3. Apply Proportional Perturbation
    # Strength: +/- 5% to 15% variation locally
    strength = np.random.uniform(0.05, 0.15) 
    
    # The perturbation follows the density shape (e.g. doughnuts)
    # New = Old * (1 + strength * random_field)
    distorted = base_density * (1.0 + (strength * smooth_field))
    
    # 4. Physics Constraint: Non-negativity
    distorted = np.maximum(distorted, 1e-12)
    
    # 5. Physics Constraint: Charge Conservation (Renormalization)
    # The sum of the density array *should* equal the number of electrons.
    current_integral = np.sum(distorted)
    
    if current_integral > 0:
        scaling_factor = target_electrons / current_integral
        distorted *= scaling_factor
        
    return distorted

def transform(density, density_ref, transform_type):
    """
    Applies the transform. 
    Crucial: For residual_noise, this subtracts the reference.
    """
    epsilon = 1e-9 
    
    if transform_type == 'value':
        return density
    elif transform_type == 'sqrt':
        return np.sqrt(np.abs(density) + epsilon)
    elif transform_type == 'log':
        return np.log(np.abs(density) + epsilon)
    elif transform_type == 'residual_noise':
        # (Density - DFT) / sqrt(DFT)
        if density_ref is None:
            raise ValueError("residual_noise requires density_ref")
        
        dval = density_ref
        dsqrt = np.sqrt(np.abs(density_ref) + epsilon)
        
        # This creates the "Standardized Noise Map"
        dtrans = (density - dval) / dsqrt
        return dtrans
    else:
        raise RuntimeError('bad transform type')

def transform_pair(x_vol, y_vol):
    """
    Applies identical random transformations to input (x) and target (y).
    """
    # 1. Random Flips
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=0)
        y_vol = np.flip(y_vol, axis=0)
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=1)
        y_vol = np.flip(y_vol, axis=1)
    if np.random.rand() > 0.5:
        x_vol = np.flip(x_vol, axis=2)
        y_vol = np.flip(y_vol, axis=2)

    # 2. Random Translation (Rolling)
    shift_x = np.random.randint(0, x_vol.shape[0])
    shift_y = np.random.randint(0, x_vol.shape[1])
    shift_z = np.random.randint(0, x_vol.shape[2])
    
    x_vol = np.roll(x_vol, shift=(shift_x, shift_y, shift_z), axis=(0, 1, 2))
    y_vol = np.roll(y_vol, shift=(shift_x, shift_y, shift_z), axis=(0, 1, 2))

    # 3. 90-Degree Rotations (XY Plane)
    k = np.random.randint(0, 4) 
    x_vol = np.rot90(x_vol, k=k, axes=(0, 1))
    y_vol = np.rot90(y_vol, k=k, axes=(0, 1))

    return x_vol, y_vol

# ==========================================
# 2. GENERATOR LOGIC (MODIFIED)
# ==========================================

def generate_dataset(ref_d, base_samples, num_train, num_val):
    """
    Generates RESIDUAL NOISE pairs with PROPORTIONAL VMC SIGNAL.
    """
    total_mc_samples = 0
    
    # Define noise levels
    input_sample_opts = [base_samples, base_samples*2, base_samples*3]

    def create_batch(n_items):
        nonlocal total_mc_samples
        x_data = np.zeros((n_items, 116, 116, 72), dtype=np.float32)
        y_data = np.zeros((n_items, 116, 116, 72), dtype=np.float32)
        
        for i in range(n_items):
            # -----------------------------------------------
            # A. CREATE "TRUE" DENSITY (DFT + Proportional Blobs)
            # -----------------------------------------------
            # We enforce the 50 electron sum here
            true_density = perturb_density(ref_d, target_electrons=50.0)
            
            # -----------------------------------------------
            # B. Generate Noisy Instances
            # -----------------------------------------------
            n_samples_in = np.random.choice(input_sample_opts)
            n_samples_out = n_samples_in * 100 
            
            total_mc_samples += (n_samples_in + n_samples_out)
            
            # The stochastic generator adds Poisson-like noise to the 
            # ALREADY perturbed density.
            raw_x = stochastic_density(true_density, n_samples_in)
            
            # Note: raw_y (Target) should ideally be the true_density itself 
            # if you want to predict the clean signal directly, OR a very clean
            # sample of it. Here we use a clean sample:
            raw_y = stochastic_density(true_density, n_samples_out)
            
            # -----------------------------------------------
            # C. Calculate Residuals vs ORIGINAL DFT
            # -----------------------------------------------
            # We verify the sum of true_density is 50 before transform
            # print(f"Debug Sum: {np.sum(true_density)}") 
            
            res_x = transform(raw_x, ref_d, 'residual_noise')
            res_y = transform(raw_y, ref_d, 'residual_noise')
            
            # -----------------------------------------------
            # D. Augment
            # -----------------------------------------------
            aug_x, aug_y = transform_pair(res_x, res_y)
            
            x_data[i] = aug_x
            y_data[i] = aug_y
            
        return x_data, y_data

    print(f"  - Generating {num_train} Training pairs (Perturbed Residuals)...")
    x_train, y_train = create_batch(num_train)
    
    print(f"  - Generating {num_val} Validation pairs (Perturbed Residuals)...")
    x_val, y_val = create_batch(num_val)

    return {
        "x_train": x_train,
        "x_val": x_val,
        "y_train": y_train,
        "y_val": y_val,
        "total_samples": total_mc_samples
    }

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

# Path to DFT Reference
dft_path = '/pscratch/sd/k/kberard/SCGSR/Data/vo2_1x1x1_opt/density_data/vmc_noJ/density_tot_ref.h5'

# Placeholder for your noise generator
def stochastic_density(density, samples):
    # DUMMY IMPLEMENTATION FOR TESTING
    # Real VMC noise scales with sqrt(density)
    noise = np.random.normal(0, 1, density.shape) * np.sqrt(np.abs(density) / samples)
    return density + noise

print(">>> Loading DFT Reference...")
with h5py.File(dft_path, 'r') as file:
    dft_d = file['density'][:] # Shape (116, 116, 72)

    # Normalize DFT input just in case it isn't exactly 50
    # This ensures our perturbations start from a valid state
    current_sum = np.sum(dft_d)
    if not np.isclose(current_sum, 50.0, atol=1e-3):
        print(f"Warning: Input DFT sum is {current_sum}. Normalizing to 50.0")
        dft_d = dft_d * (50.0 / current_sum)

# Configuration
num_train = 1000
num_val = 200
samples_list = [20000000,40000000,100000000]

for samples in samples_list:
    print(f"Generating data for base sample scale: {samples}")
    
    # Generate
    data_dict = generate_dataset(dft_d, samples, num_train, num_val)

    # Save
    file_name = f"{data_dict['total_samples']}_samples_Residual_Perturbed.npz"
    save_path = "DFT_dat_lev_" + file_name
    
    np.savez(save_path,
             x_train=data_dict["x_train"],
             x_val=data_dict["x_val"],
             y_train=data_dict["y_train"],
             y_val=data_dict["y_val"],
             total_samples=data_dict['total_samples'])

    print(f"Saved dataset to {save_path}")
    print("-" * 30)

>>> Loading DFT Reference...
Generating data for base sample scale: 20000000
  - Generating 1000 Training pairs (Perturbed Residuals)...
  - Generating 200 Validation pairs (Perturbed Residuals)...
Saved dataset to DFT_dat_lev_4819720000000_samples_Residual_Perturbed.npz
------------------------------
Generating data for base sample scale: 40000000
  - Generating 1000 Training pairs (Perturbed Residuals)...
  - Generating 200 Validation pairs (Perturbed Residuals)...
Saved dataset to DFT_dat_lev_9792960000000_samples_Residual_Perturbed.npz
------------------------------
Generating data for base sample scale: 100000000
  - Generating 1000 Training pairs (Perturbed Residuals)...
  - Generating 200 Validation pairs (Perturbed Residuals)...
Saved dataset to DFT_dat_lev_24431900000000_samples_Residual_Perturbed.npz
------------------------------
